In [ ]:
"""
AgriLST-ML : Data Preparation Pipeline STATE-WIDE PRODUCTION
================================================================
All 36 Maharashtra districts (full state coverage).
Historical window: rolling 5 years up to present (auto-computed from
run date, not hardcoded) so re-running this script later keeps pulling
"5 years to present" without manual date edits.

WORKFLOW:
  1. Upload the CLEANED CSVs to Drive (from the repair step) so resume
     logic skips already-collected dates for Wardha/Akola/Nagpur etc.
  2. Run run_pipeline() — it processes all 10 districts, skipping dates
     already in each district's CSV.
  3. Run merge_all_csvs() when all districts are done.
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS & AUTH
# ─────────────────────────────────────────────────────────────────────────────


import ee
import os
import re
import math
import time
import logging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
GEE_PROJECT = 'Your-project-ID'
ee.Authenticate()
ee.Initialize(GEE_PROJECT)


# ─────────────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('AgriLST')

def log_step(tag, msg=''): log.info(   f'▶  [{tag}] {msg}')
def log_ok  (tag, msg=''): log.info(   f'✓  [{tag}] {msg}')
def log_skip(tag, msg=''): log.warning(f'⚠  [{tag}] SKIPPED — {msg}')
def log_err (tag, msg=''): log.error(  f'✗  [{tag}] ERROR — {msg}')


# ─────────────────────────────────────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DRIVE_OUTPUT_DIR = 'DATA'

# ── All 36 Maharashtra districts — GAUL 2015 ADM2_NAME verified mapping ───
# Verified against a live query of FAO/GAUL/2015/level2 for
# ADM1_NAME='Maharashtra' (35 units returned; only Palghar is absent).
DISTRICTS_GAUL = {
    'Ahmednagar'      : 'Ahmednagar',        # GAUL spelling confirmed (was wrongly guessed 'Ahmadnagar')
    'Akola'           : 'Akola',
    'Amravati'        : 'Amravati',
    'Beed'            : 'Bid',               # GAUL uses legacy 'Bid'
    'Bhandara'        : 'Bhandara',
    'Buldhana'        : 'Buldana',           # GAUL omits the 'h'
    'Chandrapur'      : 'Chandrapur',
    'Chhatrapati Sambhajinagar': 'Aurangabad', # GAUL 2015 uses 'Aurangabad'
    'Dharashiv'       : 'Osmanabad',         # GAUL 2015 uses 'Osmanabad'
    'Dhule'           : 'Dhule',
    'Gadchiroli'      : 'Garhchiroli',       # GAUL spelling confirmed (was wrongly guessed 'Gadchiroli')
    'Gondia'          : 'Gondiya',           # GAUL spells with a 'y'
    'Hingoli'         : 'Hingoli',
    'Jalgaon'         : 'Jalgaon',
    'Jalna'           : 'Jalna',
    'Kolhapur'        : 'Kolhapur',
    'Latur'           : 'Latur',
    'Mumbai City'     : 'Mumbai city',       # GAUL keeps this SEPARATE from Suburban — confirmed, not merged
    'Mumbai Suburban' : 'Mumbai Suburban',   # GAUL keeps this SEPARATE from City — confirmed, not merged
    'Nagpur'          : 'Nagpur',
    'Nanded'          : 'Nanded',
    'Nandurbar'       : 'Nandurbar',
    'Nashik'          : 'Nashik',
    'Palghar'         : 'Thane',             # Confirmed: GAUL 2015 has no Palghar unit; falls back to parent 'Thane'
    'Parbhani'        : 'Parbhani',
    'Pune'            : 'Pune',
    'Raigad'          : 'Raigarh',           # GAUL uses legacy spelling 'Raigarh'
    'Ratnagiri'       : 'Ratnagiri',
    'Sangli'          : 'Sangli',
    'Satara'          : 'Satara',
    'Sindhudurg'      : 'Sindhudurg',
    'Solapur'         : 'Solapur',
    'Thane'           : 'Thane',
    'Wardha'          : 'Wardha',
    'Washim'          : 'Washim',
    'Yavatmal'        : 'Yavatmal',
}

DISTRICTS = list(DISTRICTS_GAUL.keys())
# ── Rolling 5-year historical window, computed from run date ─────────────────
# Re-running this script on a later date automatically shifts the window
# forward — "5 years of history up to present" rather than a fixed range.
_TODAY               = datetime.now()
START_DATE           = (_TODAY - timedelta(days=5*365)).strftime('%Y-%m-%d')
END_DATE             = _TODAY.strftime('%Y-%m-%d')
DATE_STEP_DAYS       = 8
DATE_MATCH_TOLERANCE = 5

AG_CLASS_CODES  = [10, 20, 30, 40]   # Trees, Shrubland, Grassland, Cropland
MODIS_SCALE     = 1000
TILE_SCALE      = 4

PAUSE_BETWEEN_DISTRICTS = 60   # seconds — lets GEE server settle between districts
PAUSE_BETWEEN_DATES     = 5    # seconds — prevents rate-limit errors
MAX_ROWS_PER_GETINFO    = 3000 # chunk size for direct download

# ── Quality filter thresholds (pandas-side, after download) ──────────────────
AG_FRACTION_MIN   = 0.3    # keep 1km pixels with ≥30% ag/veg cover
LST_MIN_K         = 280.0  # physically valid for Maharashtra (K)
LST_MAX_K         = 340.0  # physically valid for Maharashtra (K)
NDVI_MIN          = -0.1   # valid vegetation index lower bound
NDVI_MAX          =  0.95  # valid vegetation index upper bound
MODIS_QC_MAX_BITS = 1      # QC bits 0-1 ≤ 1: keep good(0) + marginal(1), drop cloud(2)

ERA5_FILL_K = 295.0        # fill constant for missing ERA5 air temperature

# ── CANONICAL COLUMN ORDER ────────────────────────────────────────────────────
# Every CSV write is forced through df.reindex(columns=CANONICAL_COLUMNS).
# All 24 columns appear in identical order regardless of GEE property dict order.
CANONICAL_COLUMNS = [
    'date', 'district',
    'ndvi', 'ndwi',
    'LST_modis', 'modis_qc',
    'elevation', 'slope', 'aspect_sin', 'aspect_cos',
    'soil_clay', 'soil_sand', 'soil_ph', 'soil_organic_carbon', 'soil_texture',
    'rain_0d', 'rain_5d', 'rain_10d', 'rain_15d',
    'era5_airtemp', 'era5_filled',
    'doy_sin', 'doy_cos',
    'ag_fraction'
]

SOIL_COLS = ['soil_clay', 'soil_sand', 'soil_ph', 'soil_organic_carbon', 'soil_texture']
DATE_PATTERN = re.compile(r'^\d{4}-\d{2}-\d{2}$')


# ─────────────────────────────────────────────────────────────────────────────
# 1. STATIC ASSETS  (computed once, reused across all dates and districts)
# ─────────────────────────────────────────────────────────────────────────────

def get_ag_fraction_image():
    """
    Continuous ag-fraction image (0=non-ag, 1=ag) from ESA WorldCover v200.
    NOT used as a mask here — sampled as a feature band.
    Ag filtering is applied in pandas using the ag_fraction column.
    """
    log_step('ag_frac', f'ESA WorldCover — classes: {AG_CLASS_CODES}')
    lulc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
    frac = (lulc.remap(AG_CLASS_CODES, [1]*len(AG_CLASS_CODES), 0)
            .rename('ag_fraction').toFloat())
    log_ok('ag_frac', '1=ag/veg, 0=other')
    return frac


def get_dem_features():
    """SRTM GL1 (30m): elevation, slope, aspect encoded as sin/cos."""
    log_step('dem', 'SRTM GL1 — elevation / slope / aspect')
    dem   = ee.Image('USGS/SRTMGL1_003').select('elevation')
    slope = ee.Terrain.slope(dem)
    rad   = ee.Terrain.aspect(dem).multiply(math.pi / 180.0)
    stack = (dem.rename('elevation')
             .addBands(slope.rename('slope'))
             .addBands(rad.sin().rename('aspect_sin'))
             .addBands(rad.cos().rename('aspect_cos')))
    log_ok('dem')
    return stack


def get_soil_features():
    """
    SoilGrids via OpenLandMap — surface layer (0-5cm), 250m native.
    Bands: clay%, sand%, pH×10, organic_carbon (dg/kg), texture_class (1-12).
    """
    log_step('soil', 'SoilGrids OpenLandMap 0-5cm')
    clay    = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_clay')
    sand    = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02').select('b0').rename('soil_sand')
    ph      = ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02').select('b0').rename('soil_ph')
    organic = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02').select('b0').rename('soil_organic_carbon')
    texture = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').select('b0').rename('soil_texture')
    log_ok('soil', 'clay / sand / pH / organic_carbon / texture')
    return clay.addBands([sand, ph, organic, texture])


def get_modis_projection():
    """MODIS native sinusoidal projection — used as sampling grid anchor."""
    log_step('proj', 'Fetching MODIS native projection')
    proj = (ee.ImageCollection('MODIS/061/MOD11A1')
            .filterDate('2022-01-01', '2022-01-02')
            .first().select('LST_Day_1km').projection())
    log_ok('proj', 'MODIS sinusoidal loaded')
    return proj


def get_district_aoi(district_label):
    """FeatureCollection for one district using confirmed GAUL spelling."""
    gaul_name = DISTRICTS_GAUL[district_label]
    aoi = (ee.FeatureCollection('FAO/GAUL/2015/level2')
           .filter(ee.Filter.eq('ADM2_NAME', gaul_name)))
    n = aoi.size().getInfo()
    if n == 0:
        raise ValueError(
            f'District "{district_label}" → GAUL name "{gaul_name}" '
            f'returned 0 features. Check DISTRICTS_GAUL spelling.')
    return aoi


# ─────────────────────────────────────────────────────────────────────────────
# 2. DYNAMIC FEATURES  (per date)
# ─────────────────────────────────────────────────────────────────────────────

def mask_s2_clouds(image):
    """QA60 bitmask: bit10=opaque cloud, bit11=cirrus. Both must be 0."""
    qa   = image.select('QA60')
    mask = (qa.bitwiseAnd(1 << 10).eq(0)
             .And(qa.bitwiseAnd(1 << 11).eq(0)))
    return image.updateMask(mask)


def get_s2_indices(target_date_str, aoi):
    """
    NDVI + NDWI from closest cloud-free S2 scene within ±DATE_MATCH_TOLERANCE.
    Returns None if no scene found.
    NO ag_mask applied — all land types sampled. Ag filtering done in pandas.
    """
    d   = ee.Date(target_date_str)
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(d.advance(-DATE_MATCH_TOLERANCE, 'day'),
                       d.advance( DATE_MATCH_TOLERANCE, 'day'))
           .filterBounds(aoi)
           .map(mask_s2_clouds))

    if col.size().getInfo() == 0:
        return None

    col     = col.map(lambda img: img.set('td', img.date().difference(d, 'hour').abs()))
    closest = ee.Image(col.sort('td').first())

    nir  = closest.select('B8')
    red  = closest.select('B4')
    grn  = closest.select('B3')
    swir = closest.select('B11')

    ndvi = nir.subtract(red).divide(nir.add(red)).rename('ndvi')
    ndwi = grn.subtract(swir).divide(grn.add(swir)).rename('ndwi')
    return ndvi.addBands(ndwi)


def get_modis_lst(target_date_str, aoi):
    """
    MODIS Terra LST (MOD11A1) scaled to Kelvin. QC band included as feature.
    Fast pixel-count check at 5km scale — skips fully cloud-covered dates
    immediately without building the full stack, saving ~60s per skipped date.
    Returns None if no image or all pixels are fill values.
    """
    d   = ee.Date(target_date_str)
    col = (ee.ImageCollection('MODIS/061/MOD11A1')
           .filterDate(d, d.advance(1, 'day'))
           .filterBounds(aoi))

    if col.size().getInfo() == 0:
        return None

    img = col.first()
    lst = img.select('LST_Day_1km').multiply(0.02).rename('LST_modis')
    qc  = img.select('QC_Day').rename('modis_qc')

    # Mask fill values (MODIS LST fill = 0 → 0K after ×0.02 scaling)
    lst_masked = lst.updateMask(lst.gt(0))

    # Fast check: count valid pixels at 5km — 1 round-trip, ~2 seconds
    n_valid = (lst_masked
               .reduceRegion(ee.Reducer.count(), aoi.geometry(), 5000, maxPixels=1e6)
               .get('LST_modis'))

    if ee.Number(n_valid).getInfo() == 0:
        log.warning(f'    MODIS: all pixels cloud-filled on {target_date_str} — skip')
        return None

    return lst_masked.addBands(qc)


def get_rainfall_features(target_date_str):
    """
    CHIRPS daily rainfall (mm):
    rain_0d  = same-day precipitation
    rain_5d  = 5-day rolling accumulation (short-term soil moisture response)
    rain_10d = 10-day rolling accumulation
    rain_15d = 15-day rolling accumulation (antecedent moisture state)
    These are the key features that explain why NDVI–LST slope shifts across dates.
    """
    d      = ee.Date(target_date_str)
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    def accum(days):
        return (chirps
                .filterDate(d.advance(-days, 'day'), d.advance(1, 'day'))
                .sum().rename(f'rain_{days}d'))
    return (chirps.filterDate(d, d.advance(1, 'day')).sum().rename('rain_0d')
            .addBands(accum(5)).addBands(accum(10)).addBands(accum(15)))


def get_era5_feature(target_date_str):
    """
    ERA5-Land daily 2m air temperature (Kelvin).
    FIX: Returns fill constant 295K with era5_filled=1 flag when unavailable.
    Pixel-level nulls (rare, ~<1%) are filled in pandas filter_and_save().
    """
    d   = ee.Date(target_date_str)
    col = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
           .filterDate(d, d.advance(1, 'day')))

    if col.size().getInfo() == 0:
        log.warning(f'    ERA5: no image for {target_date_str} — using fill {ERA5_FILL_K}K')
        return (ee.Image.constant(ERA5_FILL_K).rename('era5_airtemp')
                .addBands(ee.Image.constant(1).rename('era5_filled')))

    return (col.first().select('temperature_2m').rename('era5_airtemp')
            .addBands(ee.Image.constant(0).rename('era5_filled')))


def get_doy_features(target_date_str):
    """
    Day-of-year as sin/cos (cyclical encoding).
    Avoids the 365→1 discontinuity in raw integer encoding.
    sin/cos pair captures seasonal continuity correctly.
    """
    doy   = ee.Date(target_date_str).getRelative('day', 'year')
    angle = ee.Number(doy).multiply(2.0 * math.pi / 365.0)
    return (ee.Image.constant(angle.sin()).rename('doy_sin')
            .addBands(ee.Image.constant(angle.cos()).rename('doy_cos')))


# ─────────────────────────────────────────────────────────────────────────────
# 3. BUILD STACK  (no GEE-side masking — all filtering in pandas)
# ─────────────────────────────────────────────────────────────────────────────

def build_stack(target_date_str, ag_frac, dem, soil, aoi):
    """
    Assembles all 24 feature bands into one image for sampling.
    No masking applied here — sample(scale=1000) uses GEE's image pyramid
    to aggregate each band to 1km at each grid point automatically.
    Returns None if S2 or MODIS is unavailable for this date.
    """
    s2 = get_s2_indices(target_date_str, aoi)
    if s2 is None:
        return None

    lst_qc = get_modis_lst(target_date_str, aoi)
    if lst_qc is None:
        return None

    rainfall = get_rainfall_features(target_date_str)
    era5     = get_era5_feature(target_date_str)
    doy      = get_doy_features(target_date_str)

    return (s2
            .addBands(lst_qc)
            .addBands(dem)
            .addBands(soil)
            .addBands(rainfall)
            .addBands(era5)
            .addBands(doy)
            .addBands(ag_frac))


# ─────────────────────────────────────────────────────────────────────────────
# 4. SAMPLE  (dropNulls=False — all filtering in pandas)
# ─────────────────────────────────────────────────────────────────────────────

def sample_stack(stack, aoi, modis_proj, district_label, target_date_str):
    """
    Sample image stack at MODIS 1km grid.
    dropNulls=False: all pixels pass through, pandas filters after download.
    projection=modis_proj: anchors sample points to MODIS native grid.
    """
    try:
        fc = (stack.sample(
                  region=aoi.geometry(),
                  scale=MODIS_SCALE,
                  projection=modis_proj,
                  geometries=False,
                  seed=42,
                  dropNulls=False,
                  tileScale=TILE_SCALE)
              .map(lambda f: f.set({'date': target_date_str,
                                    'district': district_label})))
        return fc
    except Exception as e:
        log_err('sample', str(e))
        return None


# ─────────────────────────────────────────────────────────────────────────────
# 5. DOWNLOAD  (chunked getInfo — avoids single-call timeout)
# ─────────────────────────────────────────────────────────────────────────────

def download_fc(fc):
    """
    Download FeatureCollection to list of dicts in chunks.
    Returns [] on failure — pipeline continues to next date.
    """
    try:
        total = fc.size().getInfo()
    except Exception as e:
        log_err('download', f'size().getInfo() failed: {e}')
        return []

    if total == 0:
        return []

    log.info(f'    Downloading {total} rows ...')
    all_rows = []
    for offset in range(0, total, MAX_ROWS_PER_GETINFO):
        try:
            chunk    = ee.FeatureCollection(fc.toList(MAX_ROWS_PER_GETINFO, offset))
            features = chunk.getInfo()['features']
            rows     = [f['properties'] for f in features]
            all_rows.extend(rows)
            log.info(f'    Chunk {offset+1}–{min(offset+MAX_ROWS_PER_GETINFO, total)}/{total} ✓')
        except Exception as e:
            log_err('download', f'chunk offset={offset} failed: {e}')
            continue
    return all_rows


# ─────────────────────────────────────────────────────────────────────────────
# 6. FILTER + SAVE  ── ALL QUALITY FILTERING AND NULL HANDLING HAPPENS HERE
# ─────────────────────────────────────────────────────────────────────────────

def filter_and_save(rows, csv_path):
    """
    Quality-filter downloaded rows and append to CSV.
    Returns (n_raw, n_kept).

    Critical fixes:
    - CANONICAL_COLUMNS reindex before every write (prevents column shift)
    - ERA5 null fill (fills ~<1% null pixels with 295K fill constant)
    - Soil null fill (fills ~<0.1% edge pixels with column median)
    - Critical column existence check (skips cloud-affected dates cleanly)
    """
    if not rows:
        return 0, 0

    df    = pd.DataFrame(rows)
    n_raw = len(df)

    # ── 1. Critical column existence check ───────────────────────────────────
    # If ndvi or LST_modis are completely absent from all rows, the date had
    # full cloud cover in both sensors → all pixels null → keys absent from
    # the property dicts → column doesn't appear in DataFrame.
    missing = [c for c in ['ndvi', 'LST_modis'] if c not in df.columns]
    if missing:
        log.warning(f'    Skip: {missing} absent — full cloud cover on this date')
        return n_raw, 0

    # ── 2. Drop rows with null in critical columns ────────────────────────────
    df = df.dropna(subset=['ndvi', 'LST_modis'])
    if df.empty:
        return n_raw, 0

    # ── 3. Agricultural fraction filter ──────────────────────────────────────
    if 'ag_fraction' in df.columns:
        df = df[df['ag_fraction'] >= AG_FRACTION_MIN]

    if df.empty:
        return n_raw, 0

    # ── 4. LST physical range ─────────────────────────────────────────────────
    df = df[(df['LST_modis'] >= LST_MIN_K) & (df['LST_modis'] <= LST_MAX_K)]

    # ── 5. MODIS QC filter ────────────────────────────────────────────────────
    if 'modis_qc' in df.columns and not df.empty:
        df = df.copy()
        df['modis_qc'] = pd.to_numeric(df['modis_qc'], errors='coerce').fillna(0).astype(int)
        df = df[(df['modis_qc'].apply(lambda q: q & 3)) <= MODIS_QC_MAX_BITS]

    # ── 6. NDVI physical range ────────────────────────────────────────────────
    df = df[(df['ndvi'] >= NDVI_MIN) & (df['ndvi'] <= NDVI_MAX)]

    if df.empty:
        return n_raw, 0

    # ── 7. ERA5 null fill ────────────────────────────────────────────────
    # ~<1% of rows have null era5_airtemp (pixel-level gaps, not full image gaps).
    # Fill with ERA5_FILL_K (295K) and mark era5_filled=1.
    if 'era5_airtemp' in df.columns:
        null_mask = df['era5_airtemp'].isna()
        if null_mask.any():
            df = df.copy()
            df.loc[null_mask, 'era5_airtemp'] = ERA5_FILL_K
            if 'era5_filled' in df.columns:
                df.loc[null_mask, 'era5_filled'] = 1
    if 'era5_filled' in df.columns:
        df['era5_filled'] = df['era5_filled'].fillna(0).astype(int)

    # ── 8. Soil null fill ────────────────────────────────────────────────
    # ~<0.1% of rows at district edges have null soil values.
    # Fill with the column median of this date's batch.
    for col in SOIL_COLS:
        if col in df.columns and df[col].isna().any():
            median_val = df[col].median()
            if pd.notna(median_val):
                df = df.copy()
                df[col] = df[col].fillna(median_val)
            else:
                # If entire column is null (shouldn't happen), drop rows
                df = df.dropna(subset=[col])

    if df.empty:
        return n_raw, 0

    # ── 9. Enforce CANONICAL_COLUMNS order before writing ───────────────
    # This is the critical fix for column shift corruption.
    # reindex() places every column in the exact position defined by
    # CANONICAL_COLUMNS, filling any absent columns with NaN.
    # All 24 columns will be present in identical order in every CSV row.
    df = df.reindex(columns=CANONICAL_COLUMNS)

    # ── 10. Write to CSV ──────────────────────────────────────────────────────
    write_header = not os.path.exists(csv_path)
    df.to_csv(csv_path, mode='a', header=write_header, index=False)

    return n_raw, len(df)


# ─────────────────────────────────────────────────────────────────────────────
# 7. HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def get_csv_path(district_label):
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    return os.path.join(DRIVE_OUTPUT_DIR,
                        f'agrilst_{district_label}_{START_DATE[:4]}_{END_DATE[:4]}.csv')


def get_processed_dates(csv_path):
    """
    Read already-processed dates from an existing CSV.
    FIX: Validates date format (YYYY-MM-DD) to avoid reading corrupt
    values from old mismatch-corrupted CSVs into the skip set.
    Returns empty set if file doesn't exist or is unreadable.
    """
    if not os.path.exists(csv_path):
        return set()
    try:
        df   = pd.read_csv(csv_path, usecols=['date'], low_memory=False)
        # Keep only values that look like valid dates
        raw  = df['date'].astype(str).unique()
        done = {d for d in raw if DATE_PATTERN.match(d)}
        log.info(f'    Resume: {len(done)} valid dates already saved — skipping.')
        return done
    except Exception as e:
        log.warning(f'    Could not read existing CSV ({e}) — starting fresh.')
        return set()


def generate_date_list():
    dates, cur = [], datetime.strptime(START_DATE, '%Y-%m-%d')
    end_dt = datetime.strptime(END_DATE, '%Y-%m-%d')
    while cur <= end_dt:
        dates.append(cur.strftime('%Y-%m-%d'))
        cur += timedelta(days=DATE_STEP_DAYS)
    return dates


# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline():
    log_step('init', f'GEE project: {GEE_PROJECT}')
    log_step('output', f'Drive folder: {DRIVE_OUTPUT_DIR}')
    log_step('districts', f'Processing {len(DISTRICTS)} districts: {DISTRICTS}')
    log_step('dates', f'{START_DATE} → {END_DATE} | step:{DATE_STEP_DAYS}d | tolerance:±{DATE_MATCH_TOLERANCE}d')

    # ── Static assets — computed once ─────────────────────────────────────────
    ag_frac    = get_ag_fraction_image()
    dem        = get_dem_features()
    soil       = get_soil_features()
    modis_proj = get_modis_projection()
    date_list  = generate_date_list()
    log_step('dates', f'Total candidate dates: {len(date_list)}')
    log.info('─' * 65)

    pipeline_start = time.time()
    pipeline_total_rows = 0

    for d_idx, district_label in enumerate(DISTRICTS):
        log_step('district', f'[{d_idx+1}/{len(DISTRICTS)}] ── {district_label} ──')

        try:
            aoi = get_district_aoi(district_label)
        except ValueError as e:
            log_err('district', str(e))
            continue

        csv_path        = get_csv_path(district_label)
        processed_dates = get_processed_dates(csv_path)
        remaining       = [d for d in date_list if d not in processed_dates]

        log.info(f'    Dates total: {len(date_list)} | '
                 f'Already done: {len(processed_dates)} | '
                 f'To process: {len(remaining)}')

        if not remaining:
            log_ok('district', f'{district_label} — all dates already collected, skipping.')
            continue

        d_start         = time.time()
        dates_done      = 0
        dates_skipped   = 0
        total_rows      = 0

        for i, date_str in enumerate(remaining):
            log.info(f'  [{district_label}] {i+1:3d}/{len(remaining)} | {date_str}')

            try:
                # Build full feature + target stack
                stack = build_stack(date_str, ag_frac, dem, soil, aoi)
                if stack is None:
                    log_skip(f'{district_label}/{date_str}',
                             'S2 collection empty or MODIS all cloud-filled')
                    dates_skipped += 1
                    time.sleep(PAUSE_BETWEEN_DATES)
                    continue

                # Sample at MODIS 1km grid
                fc = sample_stack(stack, aoi, modis_proj, district_label, date_str)
                if fc is None:
                    dates_skipped += 1
                    time.sleep(PAUSE_BETWEEN_DATES)
                    continue

                # Download + filter + save to CSV
                rows          = download_fc(fc)
                n_raw, n_kept = filter_and_save(rows, csv_path)
                total_rows   += n_kept
                dates_done   += 1

                log_ok(f'{district_label}/{date_str}',
                       f'raw:{n_raw} → kept:{n_kept} | dist_total:{total_rows}')

            except Exception as e:
                log_err(f'{district_label}/{date_str}', str(e))
                dates_skipped += 1

            time.sleep(PAUSE_BETWEEN_DATES)

        pipeline_total_rows += total_rows
        elapsed = time.time() - d_start
        log_ok('district',
               f'{district_label} | processed:{dates_done} skipped:{dates_skipped} '
               f'rows_added:{total_rows} time:{elapsed/60:.1f}min | {csv_path}')
        log.info('─' * 65)

        # Pause between districts — lets GEE server settle
        if d_idx < len(DISTRICTS) - 1:
            log.info(f'  Pausing {PAUSE_BETWEEN_DISTRICTS}s before next district ...')
            time.sleep(PAUSE_BETWEEN_DISTRICTS)

    total_elapsed = time.time() - pipeline_start
    log_ok('pipeline',
           f'All districts done | '
           f'Total rows added this run: {pipeline_total_rows:,} | '
           f'Total time: {total_elapsed/60:.1f} min')
    log.info(f'CSVs saved in: {DRIVE_OUTPUT_DIR}')


# ─────────────────────────────────────────────────────────────────────────────
# 9. MERGE  (run after all districts complete)
# ─────────────────────────────────────────────────────────────────────────────

def merge_all_csvs():
    """
    Merge all district CSVs into one master training CSV.
    Enforces CANONICAL_COLUMNS order on the merged dataset.
    Prints a full quality summary.
    """
    dfs = []
    for district_label in DISTRICTS:
        path = get_csv_path(district_label)
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            # Keep only valid district rows (extra safety check)
            df = df[df['district'].isin(set(DISTRICTS_GAUL.keys()))]
            log_ok('merge', f'{district_label}: {len(df):,} rows')
            dfs.append(df)
        else:
            log_skip('merge', f'{district_label}: CSV not found at {path}')

    if not dfs:
        log_err('merge', 'No CSVs found.')
        return None

    master = pd.concat(dfs, ignore_index=True)
    master = master.reindex(columns=CANONICAL_COLUMNS)

    master_path = os.path.join(DRIVE_OUTPUT_DIR,
                               f'agrilst_MASTER_{START_DATE[:4]}_{END_DATE[:4]}.csv')
    master.to_csv(master_path, index=False)

    print(f'\n{"="*65}')
    print(f'MASTER DATASET SUMMARY')
    print(f'{"="*65}')
    print(f'Saved to:       {master_path}')
    print(f'Total rows:     {len(master):,}')
    print(f'Columns ({len(master.columns)}): {master.columns.tolist()}')
    print(f'\nRows per district:')
    dist_summary = master.groupby('district').agg(
        rows=('LST_modis', 'count'),
        dates=('date', 'nunique'),
        lst_min=('LST_modis', 'min'),
        lst_max=('LST_modis', 'max'),
        ndvi_mean=('ndvi', 'mean')
    ).round(2)
    print(dist_summary.to_string())
    print(f'\nDate range:     {master["date"].min()} → {master["date"].max()}')
    print(f'Unique dates:   {master["date"].nunique()}')
    print(f'LST (K):        {master["LST_modis"].min():.1f} – {master["LST_modis"].max():.1f}')
    print(f'NDVI:           {master["ndvi"].min():.3f} – {master["ndvi"].max():.3f}')
    print(f'\nNull counts:')
    nulls = master.isnull().sum()
    print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else '  None ✓')
    print(f'{"="*65}\n')

    return master


# ─────────────────────────────────────────────────────────────────────────────
# 10. DIAGNOSTIC  (run before pipeline to verify a district + date)
# ─────────────────────────────────────────────────────────────────────────────

def run_diagnostic(district_label='Wardha', test_date='2023-03-01'):
    """
    End-to-end sanity check for one district + one date.
    Shows: AOI, S2 availability, MODIS availability, bands, row counts,
    filter attrition, and a 5-row preview of final data.
    """
    print(f'\n{"="*65}')
    print(f'DIAGNOSTIC v8 — {district_label} | {test_date}')
    print(f'{"="*65}')

    ag_frac    = get_ag_fraction_image()
    dem        = get_dem_features()
    soil       = get_soil_features()
    modis_proj = get_modis_projection()

    try:
        aoi = get_district_aoi(district_label)
        print(f'\nAOI features:         {aoi.size().getInfo()} ✓')
    except ValueError as e:
        print(f'ERROR: {e}'); return

    s2     = get_s2_indices(test_date, aoi)
    lst_qc = get_modis_lst(test_date, aoi)

    print(f'S2 available:         {"YES ✓" if s2 else "NO ✗ — try different date"}')
    print(f'MODIS available:      {"YES ✓" if lst_qc else "NO ✗ — cloud-affected"}')

    if s2 is None or lst_qc is None:
        print(f'\nTry: run_diagnostic("{district_label}", "2023-01-01")')
        return

    stack = build_stack(test_date, ag_frac, dem, soil, aoi)
    fc    = sample_stack(stack, aoi, modis_proj, district_label, test_date)
    total = fc.size().getInfo()
    print(f'Raw rows sampled:     {total}')

    if total == 0:
        print('0 rows — check MODIS coverage for this district.'); return

    rows   = download_fc(fc.limit(300))
    df_raw = pd.DataFrame(rows)

    has_ndvi = 'ndvi' in df_raw.columns
    has_lst  = 'LST_modis' in df_raw.columns
    print(f'ndvi column:          {"YES ✓" if has_ndvi else "NO ✗ — S2 cloud-masked"}')
    print(f'LST_modis column:     {"YES ✓" if has_lst  else "NO ✗ — MODIS cloud-masked"}')

    if not has_ndvi or not has_lst:
        print(f'\nCloud-affected date. Try: run_diagnostic("{district_label}", "2023-01-01")')
        return

    # Filter attrition
    print(f'\n── Filter attrition ─────────────────────────────────')
    df = df_raw.copy()
    print(f'Raw:                       {len(df)} rows')
    df = df.dropna(subset=['ndvi', 'LST_modis'])
    print(f'After dropna(ndvi,LST):    {len(df)} rows')
    if 'ag_fraction' in df.columns:
        df = df[df['ag_fraction'] >= AG_FRACTION_MIN]
        print(f'After ag_fraction≥{AG_FRACTION_MIN}:     {len(df)} rows')
    df = df[(df['LST_modis'] >= LST_MIN_K) & (df['LST_modis'] <= LST_MAX_K)]
    print(f'After LST [{LST_MIN_K}–{LST_MAX_K}K]:  {len(df)} rows')
    if 'modis_qc' in df.columns and not df.empty:
        df = df.copy()
        df['modis_qc'] = df['modis_qc'].astype(int)
        df = df[(df['modis_qc'].apply(lambda q: q & 3)) <= MODIS_QC_MAX_BITS]
        print(f'After MODIS QC≤{MODIS_QC_MAX_BITS}:         {len(df)} rows')
    df = df[(df['ndvi'] >= NDVI_MIN) & (df['ndvi'] <= NDVI_MAX)]
    print(f'After NDVI range:          {len(df)} rows  ← FINAL')

    if df.empty:
        print('0 rows after filters — loosen AG_FRACTION_MIN or MODIS_QC_MAX_BITS.')
        return

    # Apply canonical reindex for preview
    df = df.reindex(columns=CANONICAL_COLUMNS)

    # Check nulls
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0]
    print(f'\nNull counts in final rows:')
    print('  None ✓' if null_cols.empty else null_cols.to_string())

    preview_cols = ['ndvi', 'ndwi', 'LST_modis', 'soil_clay', 'soil_ph',
                    'rain_15d', 'era5_airtemp', 'ag_fraction']
    print(f'\n── Final preview (first 5 rows) ──────────────────────')
    print(df[[c for c in preview_cols if c in df.columns]].head(5).to_string())
    print(f'\nLST:  {df["LST_modis"].min():.1f}K – {df["LST_modis"].max():.1f}K')
    print(f'NDVI: {df["ndvi"].min():.3f} – {df["ndvi"].max():.3f}')
    print(f'\n{"="*65}')
    print(f'✓ All checks passed. Safe to run run_pipeline().')
    print(f'{"="*65}\n')


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    # ── BEFORE RUNNING: upload the CLEANED CSVs to Drive ─────────────────────
    # Copy the agrilst_*_CLEAN.csv files from the repair step to:
    # /content/drive/MyDrive/TsHARP_ML_Model/DATA/
    # Rename them to remove the "_CLEAN" suffix:
    #   agrilst_Wardha_CLEAN.csv  → agrilst_Wardha_2022_2024.csv
    #   agrilst_Akola_CLEAN.csv   → agrilst_Akola_2022_2024.csv  (etc.)
    # The resume logic will then skip already-collected dates automatically.

    # STEP 1 — Diagnostic (confirm one district works before full run)
    # run_diagnostic(district_label='Wardha', test_date='2023-03-01')

    # STEP 2 — Full pipeline for all 10 districts
    # Uncomment after diagnostic passes:
    run_pipeline()

    # STEP 3 — Merge all district CSVs into master
    # Uncomment after run_pipeline() finishes:
    master_df = merge_all_csvs()
